In [2]:
import json
import requests
import paho.mqtt.client as mqtt

BROKER = "192.168.1.245"
PORT = 1883

USERNAME = "emonpi"
PASSWORD = "emonpi"

# MQTT TOPICS
TOPIC_CAR = "rover/power"
TOPIC_SOLAR = "rover/solar"

# SOLAR MAX CAPACITY (W)
SOLAR_MAX = 4000

# ABLY
ABLY_URL = "https://rest.ably.io/channels/nova/messages"
ABLY_KEY = "CClXdw.Z3P7Fw:G1W_WXLZYUpqqnjvplbv_GDmUJ3TB4lk1bs54DblqpE"

# LIVE VALUES
state = {
    "car": 0,
    "solar": 0,
    "solar_pct": 0
}


def send_to_ably():

    payload = {
        "name": "mqtt",
        "data": state
    }

    response = requests.post(
        ABLY_URL,
        auth=tuple(ABLY_KEY.split(":")),
        headers={
            "Content-Type": "application/json"
        },
        data=json.dumps(payload)
    )

    print(
        "Sent:",
        payload,
        "STATUS:",
        response.status_code
    )


def on_connect(client, userdata, flags, rc):

    if rc == 0:

        print("Connected to MQTT")

        client.subscribe(TOPIC_CAR)
        client.subscribe(TOPIC_SOLAR)

        print("Subscribed")

    else:

        print("Connection failed:", rc)


def on_message(client, userdata, msg):

    try:

        value = float(
            msg.payload.decode().strip()
        )

        print(msg.topic, "=", value)

        # CAR
        if msg.topic == TOPIC_CAR:

            state["car"] = int(value)

        # SOLAR
        elif msg.topic == TOPIC_SOLAR:

            solar = int(value)

            pct = int(
                max(
                    0,
                    min(
                        100,
                        (solar / SOLAR_MAX) * 100
                    )
                )
            )

            state["solar"] = solar
            state["solar_pct"] = pct

        send_to_ably()

    except Exception as e:

        print("Error:", e)


client = mqtt.Client()

client.username_pw_set(
    USERNAME,
    PASSWORD
)

client.on_connect = on_connect
client.on_message = on_message

client.connect(
    BROKER,
    PORT,
    60
)

client.loop_forever()

/tmp/ipykernel_6443/231240203.py:111: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  client = mqtt.Client()


Connected to MQTT
Subscribed
rover/power = 0.0
Sent: {'name': 'mqtt', 'data': {'car': 0, 'solar': 0, 'solar_pct': 0}} STATUS: 201
rover/solar = 500.0
Sent: {'name': 'mqtt', 'data': {'car': 0, 'solar': 500, 'solar_pct': 12}} STATUS: 201


KeyboardInterrupt: 